# Notebook 8: SVM Multi-Class Classification

**Difficulty:** ⭐⭐⭐ | **Estimated Time:** 2.5 hours | **Week 10 — Support Vector Machines & Kernel Methods**

---

## Why This Matters

Binary classifiers are everywhere in ML theory, but real-world problems almost never have just two classes. Your email inbox contains spam, legitimate email, promotional messages, and newsletters — not just spam/not-spam. Knowing how to extend SVMs (which are fundamentally binary) to multi-class problems is an essential engineering skill.

The two dominant strategies — One-vs-Rest and One-vs-One — represent a fundamental tradeoff between simplicity and quality. Understanding them from scratch will help you make principled choices (and debug sklearn's `SVC` when it behaves unexpectedly).

---

## Real-World Analogy: Tournament Brackets

Imagine determining which of four email types (spam, ham, promotional, newsletter) best matches an incoming message — it's a **tournament**:

**One-vs-One (OvO) — Round-Robin Tournament:**  
Hold every possible pairwise match: spam vs ham, spam vs promo, spam vs newsletter, ham vs promo, ham vs newsletter, promo vs newsletter. That's C(4,2) = **6 matches**. Each match is decided by a binary SVM. When all matches are done, you crown the email type with the **most wins**.

**One-vs-Rest (OvR) — Four-Way Decathlon:**  
Each type competes in its own event against all others: "spam vs (everything else)", "ham vs (everything else)", etc. The winner is whichever type wins its individual event **most confidently** (highest decision function score).

**DAGSVM — Elimination Bracket:**  
Uses OvO classifiers arranged in a tree: spam is eliminated first if it loses to ham, the winner then faces promo, etc. More efficient at test time.

---

## Learning Objectives

By the end of this notebook you will be able to:
1. Explain why SVMs are natively binary and why a multi-class strategy is needed
2. Implement One-vs-Rest (OvR) from scratch using 4 binary SVMs
3. Implement One-vs-One (OvO) from scratch using 6 binary SVMs
4. Compare OvR, OvO, and sklearn's SVC using classification reports
5. Diagnose which class pairs are confused most by the binary sub-classifiers
6. Reason about scalability: classifier count, training cost, and test-time cost

## Section 0 — Setup & Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from itertools import combinations

from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.multiclass import OneVsRestClassifier, OneVsOneClassifier

np.random.seed(42)
sns.set_theme(style='whitegrid')

# Class labels matching our email theme
CLASS_NAMES  = ['Spam', 'Ham', 'Promotional', 'Newsletter']
CLASS_COLORS = ['#E63946', '#457B9D', '#2A9D8F', '#E9C46A']

print('Libraries loaded.')
print(f'Classes: {CLASS_NAMES}')

## Section 1 — Concept: Why SVMs Are Natively Binary

The SVM optimisation problem is built for **two classes**:

$$\min_{\mathbf{w}, b, \boldsymbol{\xi}} \frac{1}{2}\|\mathbf{w}\|^2 + C\sum_i\xi_i$$
$$\text{subject to: } y_i(\mathbf{w}\cdot\mathbf{x}_i + b) \geq 1 - \xi_i, \quad y_i \in \{-1, +1\}$$

The labels y_i must be ±1 — exactly two classes. There's no natural extension of the margin concept to 3+ classes within the primal formulation (unlike softmax for neural networks).

**Multi-class strategies:**

| Strategy | # Classifiers | Training Data per Clf | Test-time Cost |
|----------|--------------|----------------------|----------------|
| One-vs-Rest (OvR) | K | All n samples (1 vs n-n/K) | K evaluations |
| One-vs-One (OvO)  | K(K-1)/2 | ~2n/K samples per pair | K(K-1)/2 evaluations |
| DAGSVM            | K(K-1)/2 | ~2n/K samples per pair | K-1 evaluations |

For our 4-class email problem:
- OvR: **4** binary SVMs
- OvO: **6** binary SVMs  
- DAGSVM: 6 classifiers, but only 3 evaluations at test time

## Section 2 — Creating the 4-Class Email Dataset

We'll create a synthetic 2D dataset with 4 email classes, using features that mimic real email signals:
- **Feature 1 (`word_count`):** Total word count of the email (normalised)
- **Feature 2 (`spam_ratio`):** Ratio of flagged words to total words (normalised)

The 2D space lets us visualise all decision boundaries clearly.

In [ ]:
np.random.seed(42)

n_per_class = 100  # 400 total samples

# Class centres in (word_count_normalised, spam_ratio_normalised) space
# Deliberately placing them to create some overlap between adjacent classes
centres = {
    'Spam':        {'mean': [2.5,  2.5],  'cov': [[0.8, 0.3], [0.3, 0.8]]},
    'Ham':         {'mean': [-2.5, -2.5], 'cov': [[0.9, -0.2], [-0.2, 0.9]]},
    'Promotional': {'mean': [2.5,  -2.0], 'cov': [[1.0, 0.1], [0.1, 0.7]]},
    'Newsletter':  {'mean': [-2.0,  2.0], 'cov': [[0.7, 0.2], [0.2, 1.0]]},
}

X_list, y_list = [], []
for class_idx, (class_name, params) in enumerate(centres.items()):
    samples = np.random.multivariate_normal(
        params['mean'], params['cov'], n_per_class
    )
    X_list.append(samples)
    y_list.extend([class_idx] * n_per_class)

X = np.vstack(X_list)
y = np.array(y_list)

# Scale features
scaler = StandardScaler()
X = scaler.fit_transform(X)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print(f'Total samples:  {len(X)}')
print(f'Training:       {len(X_train)}')
print(f'Test:           {len(X_test)}')
print(f'Features:       {X.shape[1]} (word_count_norm, spam_ratio_norm)')
print(f'Classes:        {CLASS_NAMES}')

# Visualise the dataset
fig, ax = plt.subplots(1, 1, figsize=(7, 6))
for c_idx, (c_name, c_color) in enumerate(zip(CLASS_NAMES, CLASS_COLORS)):
    mask = y == c_idx
    ax.scatter(X[mask, 0], X[mask, 1],
               color=c_color, edgecolors='k', linewidths=0.4,
               s=50, alpha=0.85, label=c_name)

ax.set_xlabel('word_count (normalised)', fontsize=12)
ax.set_ylabel('spam_ratio (normalised)', fontsize=12)
ax.set_title('4-Class Email Dataset\n(Spam, Ham, Promotional, Newsletter)',
             fontsize=12, fontweight='bold')
ax.legend(title='Email Class', fontsize=10)
plt.tight_layout()
plt.show()

## Section 3 — One-vs-Rest (OvR): Manual Implementation

**Strategy:** Train K = 4 binary SVMs. For class k, label all class-k samples as +1 and all others as -1.  
At prediction time, run all K classifiers and pick the one with the highest **decision function score** (confidence).

**Key insight:** The raw SVM output (before the ±1 threshold) is a signed distance to the hyperplane. A larger positive value = more confidently in the positive class. OvR uses these raw scores to break ties.

In [ ]:
# ── Train 4 binary OvR classifiers ─────────────────────────────────────────
ovr_classifiers = []
ovr_train_info = []

for k in range(len(CLASS_NAMES)):
    # Binarise labels: class k = +1, everything else = -1
    y_binary = np.where(y_train == k, 1, -1)

    n_pos = (y_binary == 1).sum()
    n_neg = (y_binary == -1).sum()

    clf = SVC(kernel='rbf', C=1.0, gamma='scale', probability=False)
    clf.fit(X_train, y_binary)
    ovr_classifiers.append(clf)

    # Per-classifier training accuracy
    y_pred_binary = clf.predict(X_train)
    bin_acc = accuracy_score(y_binary, y_pred_binary)
    ovr_train_info.append({
        'Class': CLASS_NAMES[k],
        'Positive samples': n_pos,
        'Negative samples': n_neg,
        'Imbalance ratio': round(n_neg / n_pos, 1),
        'Binary train acc': round(bin_acc, 3),
        'n_support_vectors': clf.n_support_.sum()
    })
    print(f'Trained OvR classifier {k}: {CLASS_NAMES[k]:12s} vs Rest | '
          f'n_pos={n_pos}, n_neg={n_neg} | acc={bin_acc:.3f}')

print()
df_ovr_info = pd.DataFrame(ovr_train_info)
print(df_ovr_info.to_string(index=False))

print()
print('Note: Each OvR classifier has n_pos vs 3*n_pos = 1:3 class imbalance.')
print('This imbalance can distort the decision boundaries and bias predictions.')

## Section 4 — OvR Prediction Function & Evaluation

In [ ]:
def ovr_predict(classifiers, X):
    """
    OvR prediction: for each sample, choose the class whose binary
    classifier returns the highest decision function value.

    Args:
        classifiers: list of K fitted binary SVC objects
        X: array of shape (n_samples, n_features)

    Returns:
        predicted class indices of shape (n_samples,)
    """
    # Each clf.decision_function returns (n_samples,) — the signed distance
    decision_values = np.array([
        clf.decision_function(X) for clf in classifiers
    ])  # shape: (K, n_samples)

    # The class with the highest score wins
    return np.argmax(decision_values, axis=0)  # shape: (n_samples,)


def ovr_decision_matrix(classifiers, X):
    """Return the full (n_samples, K) decision function matrix."""
    return np.array([clf.decision_function(X) for clf in classifiers]).T


# Evaluate on test set
y_pred_ovr = ovr_predict(ovr_classifiers, X_test)
ovr_acc = accuracy_score(y_test, y_pred_ovr)

print(f'OvR Test Accuracy: {ovr_acc:.4f}')
print()
print('OvR Classification Report:')
print(classification_report(y_test, y_pred_ovr, target_names=CLASS_NAMES))

## Section 5 — Visualising OvR Decision Boundaries

We'll show two types of plots:
1. **Four individual binary boundaries** — one for each OvR classifier  
2. **The combined multi-class boundary** — the result of `argmax` over all four

In [ ]:
# Grid for plotting
x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300),
                     np.linspace(y_min, y_max, 300))
grid = np.c_[xx.ravel(), yy.ravel()]

fig = plt.figure(figsize=(18, 12))
fig.suptitle('One-vs-Rest (OvR): Individual Binary Boundaries + Combined Multi-Class Boundary',
             fontsize=13, fontweight='bold')

# 2×3 grid: first 4 = individual OvR, last 2: combined + confusion matrix
gs = fig.add_gridspec(2, 3, hspace=0.4, wspace=0.3)

# Individual binary boundaries
for k in range(4):
    row, col = k // 3, k % 3
    ax = fig.add_subplot(gs[row, col])

    # Binary decision: +1 (is this class) vs -1 (not this class)
    Z_bin = ovr_classifiers[k].predict(grid).reshape(xx.shape)
    ax.contourf(xx, yy, Z_bin, alpha=0.25, cmap='PiYG', levels=[-1.5, 0, 1.5])
    ax.contour(xx, yy, Z_bin, colors='k', linewidths=1.0, levels=[0])

    # Plot all 4 classes
    for c_idx, (c_name, c_color) in enumerate(zip(CLASS_NAMES, CLASS_COLORS)):
        mask = y_train == c_idx
        marker = 'o' if c_idx == k else 'x'
        size   = 50 if c_idx == k else 20
        alpha  = 0.9 if c_idx == k else 0.4
        ax.scatter(X_train[mask, 0], X_train[mask, 1],
                   color=c_color, marker=marker, s=size, alpha=alpha,
                   edgecolors='k' if marker == 'o' else 'none', linewidths=0.3,
                   label=c_name if c_idx == k else None)

    ax.set_title(f'OvR Classifier {k}: {CLASS_NAMES[k]} vs Rest\n'
                 f'(green = positive / "{CLASS_NAMES[k]}" region)',
                 fontsize=9, fontweight='bold')
    ax.set_xlim(x_min, x_max); ax.set_ylim(y_min, y_max)
    ax.set_xlabel('word_count'); ax.set_ylabel('spam_ratio')

# Combined multi-class boundary
ax_combined = fig.add_subplot(gs[1, 1])
Z_multi = ovr_predict(ovr_classifiers, grid).reshape(xx.shape)
cmap_4  = plt.cm.colors.ListedColormap(CLASS_COLORS)
ax_combined.contourf(xx, yy, Z_multi, alpha=0.3, cmap=cmap_4, levels=[-0.5, 0.5, 1.5, 2.5, 3.5])
ax_combined.contour(xx, yy, Z_multi, colors='k', linewidths=0.8)
for c_idx, (c_name, c_color) in enumerate(zip(CLASS_NAMES, CLASS_COLORS)):
    mask = y == c_idx
    ax_combined.scatter(X[mask, 0], X[mask, 1], color=c_color, edgecolors='k',
                        linewidths=0.3, s=25, alpha=0.9, label=c_name)
ax_combined.set_title(f'Combined OvR Multi-Class Boundary\nTest Acc: {ovr_acc:.3f}',
                      fontsize=10, fontweight='bold')
ax_combined.legend(fontsize=7, title='Class')
ax_combined.set_xlabel('word_count'); ax_combined.set_ylabel('spam_ratio')

# Confusion matrix
ax_cm = fig.add_subplot(gs[1, 2])
cm = confusion_matrix(y_test, y_pred_ovr)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax_cm,
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            linewidths=0.5, linecolor='grey')
ax_cm.set_xlabel('Predicted', fontsize=10); ax_cm.set_ylabel('True', fontsize=10)
ax_cm.set_title('OvR Confusion Matrix\n(Test Set)', fontsize=10, fontweight='bold')
plt.setp(ax_cm.get_xticklabels(), rotation=30, ha='right', fontsize=8)
plt.setp(ax_cm.get_yticklabels(), rotation=0, fontsize=8)

plt.show()

## Section 6 — One-vs-One (OvO): Manual Implementation

**Strategy:** Train one binary SVM for every pair of classes.  
For K=4 classes: C(4,2) = 4!/(2!×2!) = **6 binary SVMs**.  
Each classifier is trained only on samples from its two classes — no imbalance problem!  
Prediction: **majority vote** — whichever class gets the most wins.

**Pairs for K=4:**  
(Spam vs Ham), (Spam vs Promo), (Spam vs Newsletter), (Ham vs Promo), (Ham vs Newsletter), (Promo vs Newsletter)

In [ ]:
# ── Generate all C(K,2) pairs ────────────────────────────────────────────────
K = len(CLASS_NAMES)
pairs = list(combinations(range(K), 2))  # [(0,1), (0,2), (0,3), (1,2), (1,3), (2,3)]

print(f'K = {K} classes => C({K},2) = {len(pairs)} binary OvO classifiers')
print('Pairs:', [(CLASS_NAMES[i], CLASS_NAMES[j]) for i, j in pairs])
print()

# ── Train 6 binary OvO classifiers ──────────────────────────────────────────
ovo_classifiers = {}   # key: (i, j) -> fitted SVC
ovo_train_info = []

for (i, j) in pairs:
    # Select only samples from class i and class j
    mask = (y_train == i) | (y_train == j)
    X_pair = X_train[mask]
    y_pair = y_train[mask]

    # Convert to binary labels: class i = +1, class j = -1
    y_binary = np.where(y_pair == i, 1, -1)

    clf = SVC(kernel='rbf', C=1.0, gamma='scale')
    clf.fit(X_pair, y_binary)
    ovo_classifiers[(i, j)] = clf

    bin_acc = accuracy_score(y_binary, clf.predict(X_pair))
    ovo_train_info.append({
        'Pair': f'{CLASS_NAMES[i]} vs {CLASS_NAMES[j]}',
        'n_train': len(X_pair),
        'n_class_i': (y_binary == 1).sum(),
        'n_class_j': (y_binary == -1).sum(),
        'Binary Train Acc': round(bin_acc, 3),
        'n_SVs': clf.n_support_.sum()
    })
    print(f'  ({CLASS_NAMES[i]:12s} vs {CLASS_NAMES[j]:12s}): '
          f'n_train={len(X_pair)}, acc={bin_acc:.3f}, SVs={clf.n_support_.sum()}')

print()
df_ovo = pd.DataFrame(ovo_train_info)
print(df_ovo.to_string(index=False))
print()
print('Note: Each OvO classifier is trained on a BALANCED 50/50 subset.')
print('This is OvO\'s key advantage over OvR — no class imbalance.')

## Section 7 — OvO Prediction (Majority Voting) & Evaluation

In [ ]:
def ovo_predict(classifiers, pairs, K, X):
    """
    OvO prediction via majority vote.

    For each sample:
      - Run all C(K,2) binary classifiers
      - The winning class in each match gets 1 vote
      - Return the class with the most votes (argmax of vote tallies)

    Tie-breaking: ties are broken by summing decision function values
    (sklearn uses a similar strategy internally).

    Args:
        classifiers: dict {(i,j): SVC} for each pair
        pairs:       list of (i, j) tuples
        K:           number of classes
        X:           (n_samples, n_features)

    Returns:
        predicted labels (n_samples,)
    """
    n_samples = X.shape[0]
    votes = np.zeros((n_samples, K), dtype=int)

    for (i, j) in pairs:
        clf = classifiers[(i, j)]
        binary_pred = clf.predict(X)   # +1 (class i wins) or -1 (class j wins)
        # Add votes: +1 prediction means class i gets a vote; -1 means class j
        votes[binary_pred == 1, i] += 1
        votes[binary_pred == -1, j] += 1

    return np.argmax(votes, axis=1)


def ovo_vote_matrix(classifiers, pairs, K, X):
    """Return full (n_samples, K) vote count matrix."""
    n_samples = X.shape[0]
    votes = np.zeros((n_samples, K), dtype=int)
    for (i, j) in pairs:
        clf = classifiers[(i, j)]
        binary_pred = clf.predict(X)
        votes[binary_pred == 1, i] += 1
        votes[binary_pred == -1, j] += 1
    return votes


# Evaluate
y_pred_ovo = ovo_predict(ovo_classifiers, pairs, K, X_test)
ovo_acc = accuracy_score(y_test, y_pred_ovo)

print(f'OvO Test Accuracy: {ovo_acc:.4f}')
print()
print('OvO Classification Report:')
print(classification_report(y_test, y_pred_ovo, target_names=CLASS_NAMES))

## Section 8 — Sklearn SVC Comparison

sklearn's `SVC` uses OvO internally by default (regardless of `decision_function_shape`).  
Let's confirm our manual implementations match sklearn's output.

In [ ]:
# sklearn's native multi-class SVC (internally uses OvO)
sk_svc = SVC(kernel='rbf', C=1.0, gamma='scale', decision_function_shape='ovr')
sk_svc.fit(X_train, y_train)
y_pred_sk = sk_svc.predict(X_test)
sk_acc = accuracy_score(y_test, y_pred_sk)

# sklearn's explicit OvR and OvO wrappers
sk_ovr = OneVsRestClassifier(SVC(kernel='rbf', C=1.0, gamma='scale'))
sk_ovr.fit(X_train, y_train)
sk_ovr_acc = accuracy_score(y_test, sk_ovr.predict(X_test))

sk_ovo = OneVsOneClassifier(SVC(kernel='rbf', C=1.0, gamma='scale'))
sk_ovo.fit(X_train, y_train)
sk_ovo_acc = accuracy_score(y_test, sk_ovo.predict(X_test))

# Summary comparison
comparison = {
    'Method':   ['Manual OvR',   'Manual OvO',   'sklearn SVC',   'sklearn OvR',   'sklearn OvO'],
    'Acc':      [ovr_acc,        ovo_acc,        sk_acc,          sk_ovr_acc,      sk_ovo_acc],
    '# Classifiers': [K,        len(pairs),     len(pairs),       K,               len(pairs)],
    'Strategy': ['OvR',         'OvO',          'OvO (default)',  'OvR',           'OvO'],
}
df_compare = pd.DataFrame(comparison)
df_compare['Acc'] = df_compare['Acc'].round(4)
print('Comparison of Multi-Class SVM Methods:')
print(df_compare.to_string(index=False))

# Classification reports side by side
print()
print('=' * 60)
print('sklearn SVC (native) Classification Report:')
print(classification_report(y_test, y_pred_sk, target_names=CLASS_NAMES))

print('=' * 60)
print('sklearn OneVsOneClassifier Report:')
print(classification_report(y_test, sk_ovo.predict(X_test), target_names=CLASS_NAMES))

## Section 9 — Decision Function Values for 20 Test Points

For OvR, each of the 4 classifiers outputs a **confidence score** (signed distance to its hyperplane).  
The class with the highest score wins. This plot makes that explicit for 20 test samples.

In [ ]:
# Select 20 test samples (5 from each class for balanced display)
np.random.seed(42)
sample_indices = []
for c in range(K):
    class_indices = np.where(y_test == c)[0]
    sample_indices.extend(np.random.choice(class_indices, size=5, replace=False))
sample_indices = np.array(sample_indices)

X_sample = X_test[sample_indices]
y_sample = y_test[sample_indices]

# Get OvR decision function values — shape: (20, 4)
dec_matrix = ovr_decision_matrix(ovr_classifiers, X_sample)
y_pred_sample = np.argmax(dec_matrix, axis=1)

fig, axes = plt.subplots(4, 1, figsize=(14, 14), sharex=True)
fig.suptitle('OvR Decision Function Values for 20 Test Samples\n'
             '(Winning class = highest bar per sample | Correct = green border | Wrong = red border)',
             fontsize=12, fontweight='bold')

sample_labels = [f'S{i+1}\n({CLASS_NAMES[y_sample[i]][:4]})' for i in range(20)]
x_pos = np.arange(20)

for c_idx, (ax, c_name, c_color) in enumerate(zip(axes, CLASS_NAMES, CLASS_COLORS)):
    scores = dec_matrix[:, c_idx]
    bars = ax.bar(x_pos, scores, color=c_color, edgecolor='k', linewidth=0.5, alpha=0.8)

    # Highlight winning bars and add border for correctness
    for s_idx, (bar, score) in enumerate(zip(bars, scores)):
        is_winner  = (y_pred_sample[s_idx] == c_idx)
        is_correct = (y_pred_sample[s_idx] == y_sample[s_idx])
        if is_winner:
            bar.set_edgecolor('#2A9D8F' if is_correct else '#E63946')
            bar.set_linewidth(2.5)
            bar.set_alpha(1.0)

    ax.axhline(0, color='k', linewidth=0.8)
    ax.set_ylabel(f'{c_name}\nscore', fontsize=9)
    ax.set_xticks(x_pos)
    if c_idx == 3:
        ax.set_xticklabels(sample_labels, fontsize=7)
    ax.set_title(f'Classifier {c_idx}: "{c_name} vs Rest" — scores > 0 predict this class',
                 fontsize=9)

# Add class separation lines
for ax in axes:
    for sep in [4.5, 9.5, 14.5]:
        ax.axvline(sep, color='purple', linewidth=1.0, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

# Print prediction table
print('\nDecision function scores for 20 samples (row = sample, col = class score):')
df_dec = pd.DataFrame(dec_matrix.round(3), columns=CLASS_NAMES)
df_dec.insert(0, 'True',      [CLASS_NAMES[y] for y in y_sample])
df_dec.insert(1, 'Predicted', [CLASS_NAMES[p] for p in y_pred_sample])
df_dec.insert(2, 'Correct',   ['YES' if y == p else 'NO' for y, p in zip(y_sample, y_pred_sample)])
print(df_dec.to_string(index=False))

## Section 10 — Confusing Class Pairs: Which OvO Classifiers Fail Most?

In [ ]:
# Evaluate each OvO binary classifier on test samples from its two classes
pair_errors = []

for (i, j) in pairs:
    # Test samples from only class i and class j
    mask = (y_test == i) | (y_test == j)
    X_pair_test = X_test[mask]
    y_pair_test = y_test[mask]
    y_binary_test = np.where(y_pair_test == i, 1, -1)

    clf = ovo_classifiers[(i, j)]
    y_pred_pair = clf.predict(X_pair_test)
    acc = accuracy_score(y_binary_test, y_pred_pair)
    error_rate = 1 - acc
    n_errors = (y_binary_test != y_pred_pair).sum()

    pair_errors.append({
        'Pair': f'{CLASS_NAMES[i]} vs {CLASS_NAMES[j]}',
        'i': i, 'j': j,
        'n_test': len(X_pair_test),
        'Accuracy': round(acc, 3),
        'Error Rate': round(error_rate, 3),
        'n_errors': n_errors
    })

df_errors = pd.DataFrame(pair_errors).sort_values('Error Rate', ascending=False)
print('OvO Binary Classifier Performance on Test Pairs (sorted by error rate):')
print(df_errors[['Pair', 'n_test', 'Accuracy', 'Error Rate', 'n_errors']].to_string(index=False))

# Visualise as heatmap
error_matrix = np.zeros((K, K))
acc_matrix   = np.zeros((K, K))
for row in pair_errors:
    i, j = row['i'], row['j']
    error_matrix[i, j] = row['Error Rate']
    error_matrix[j, i] = row['Error Rate']
    acc_matrix[i, j]   = row['Accuracy']
    acc_matrix[j, i]   = row['Accuracy']

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.heatmap(acc_matrix, annot=True, fmt='.2f', cmap='YlGn',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            ax=axes[0], vmin=0.5, vmax=1.0,
            linewidths=0.5, linecolor='grey', mask=np.eye(K, dtype=bool))
axes[0].set_title('OvO Pairwise Accuracy\n(diagonal masked — not applicable)',
                  fontsize=10, fontweight='bold')

sns.heatmap(error_matrix, annot=True, fmt='.2f', cmap='Reds',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            ax=axes[1], vmin=0.0, vmax=0.5,
            linewidths=0.5, linecolor='grey', mask=np.eye(K, dtype=bool))
axes[1].set_title('OvO Pairwise Error Rate\n(higher = harder to separate)',
                  fontsize=10, fontweight='bold')

for ax in axes:
    plt.setp(ax.get_xticklabels(), rotation=30, ha='right', fontsize=9)
    plt.setp(ax.get_yticklabels(), rotation=0, fontsize=9)

plt.tight_layout()
plt.show()

print('\nHighest error pair = classes that are hardest to separate.')
hardest = df_errors.iloc[0]
print(f'Hardest pair: {hardest["Pair"]} (error rate = {hardest["Error Rate"]:.3f})')
print('This pair likely has the most overlap in feature space.')

## Section 11 — Scalability: 10-Class Comparison

The cost difference between OvR and OvO grows dramatically as K increases.  
For email classification with 10 categories (spam, ham, promo, newsletters, social, updates, forums, receipts, alerts, drafts):

In [ ]:
import time
from math import comb

# Theoretical scalability analysis
K_values = list(range(2, 21))
ovr_count = [k for k in K_values]
ovo_count = [comb(k, 2) for k in K_values]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('OvR vs OvO Scalability: Number of Binary Classifiers Required',
             fontsize=12, fontweight='bold')

ax1 = axes[0]
ax1.plot(K_values, ovr_count, 'o-', color='#E63946', linewidth=2, label='OvR (K classifiers)', markersize=6)
ax1.plot(K_values, ovo_count, 's-', color='#457B9D', linewidth=2, label='OvO (K(K-1)/2)', markersize=6)
ax1.set_xlabel('Number of Classes (K)', fontsize=11)
ax1.set_ylabel('Number of Binary Classifiers', fontsize=11)
ax1.set_title('Linear vs Quadratic Growth', fontsize=11, fontweight='bold')
ax1.legend(fontsize=10)
ax1.axvline(4, color='green', linestyle='--', alpha=0.7, label='Our email problem (K=4)')
ax1.axvline(10, color='purple', linestyle='--', alpha=0.7, label='10-class email')
ax1.legend(fontsize=9)

# Training time comparison (empirical on 10-class problem)
np.random.seed(42)
K10 = 10
n10 = 500  # 50 per class
X10 = np.random.randn(n10, 2)
y10 = np.repeat(np.arange(K10), n10 // K10)

methods = ['OvR (K=10)', 'OvO (K=10)']
clf_counts = [K10, comb(K10, 2)]
times = []

for clf_class in [OneVsRestClassifier, OneVsOneClassifier]:
    clf = clf_class(SVC(kernel='rbf', C=1.0, gamma='scale'))
    t0 = time.time()
    clf.fit(X10, y10)
    times.append(time.time() - t0)

ax2 = axes[1]
bars = ax2.bar(methods, times, color=['#E63946', '#457B9D'], edgecolor='k', linewidth=0.8)
for bar, t, n_clf in zip(bars, times, clf_counts):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
             f'{t:.4f}s\n({n_clf} classifiers)', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax2.set_ylabel('Training Time (seconds)', fontsize=11)
ax2.set_title(f'Training Time: K=10 Email Classification\n(n={n10} samples, RBF kernel)',
              fontsize=10, fontweight='bold')
ax2.set_ylim(0, max(times) * 1.5)

plt.tight_layout()
plt.show()

# Printed summary table
print('Classifier count comparison:')
print(f'{'K':>4} | {'OvR classifiers':>16} | {'OvO classifiers':>16} | {'OvO/OvR ratio':>15}')
print('-' * 58)
for k in [2, 3, 4, 5, 10, 15, 20, 50, 100]:
    n_ovr = k
    n_ovo = comb(k, 2)
    print(f'{k:>4} | {n_ovr:>16,} | {n_ovo:>16,} | {n_ovo/n_ovr:>15.1f}x')

print()
print('Key observations:')
print('  - OvO grows quadratically: O(K²) classifiers')
print('  - OvR grows linearly:      O(K)  classifiers')
print('  - But each OvO classifier trains on only 2/K of the data')
print('  - Net training cost OvO ≈ O(K²) × O(n/K) = O(nK) = same order as OvR')
print('  - Test time OvO = O(K²) evaluations vs OvR = O(K) evaluations')
print('  - For K=100: OvO requires 4950 classifiers — test time becomes expensive')

## Section 12 — OvR vs OvO: Combined Visualisation

Final side-by-side comparison of our manual implementations vs sklearn, with confusion matrices.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
fig.suptitle('Multi-Class SVM: OvR vs OvO — Decision Boundaries & Confusion Matrices',
             fontsize=13, fontweight='bold')

methods_grid = [
    ('Manual OvR', ovr_predict(ovr_classifiers, grid), ovr_acc, y_pred_ovr),
    ('Manual OvO', ovo_predict(ovo_classifiers, pairs, K, grid), ovo_acc, y_pred_ovo),
    ('sklearn SVC (native)', sk_svc.predict(grid), sk_acc, y_pred_sk),
]

# Custom colourmap
from matplotlib.colors import ListedColormap
cmap4 = ListedColormap(CLASS_COLORS)

for col, (method_name, grid_pred, acc, y_pred_method) in enumerate(methods_grid):
    # Decision boundary plot
    ax_db = axes[0, col]
    Z = grid_pred.reshape(xx.shape)
    ax_db.contourf(xx, yy, Z, alpha=0.3, cmap=cmap4, levels=[-0.5, 0.5, 1.5, 2.5, 3.5])
    ax_db.contour(xx, yy, Z, colors='k', linewidths=0.7)
    for c_idx, (c_name, c_color) in enumerate(zip(CLASS_NAMES, CLASS_COLORS)):
        mask_tr = y_train == c_idx
        mask_te = y_test == c_idx
        ax_db.scatter(X_train[mask_tr, 0], X_train[mask_tr, 1],
                      color=c_color, marker='o', s=30, alpha=0.6,
                      edgecolors='k', linewidths=0.3)
        ax_db.scatter(X_test[mask_te, 0], X_test[mask_te, 1],
                      color=c_color, marker='*', s=70, alpha=0.9,
                      edgecolors='k', linewidths=0.4)
    ax_db.set_title(f'{method_name}\nTest Acc: {acc:.3f}',
                    fontsize=10, fontweight='bold')
    ax_db.set_xlabel('word_count'); ax_db.set_ylabel('spam_ratio')
    ax_db.set_xlim(x_min, x_max); ax_db.set_ylim(y_min, y_max)

    if col == 0:
        legend_patches = [mpatches.Patch(color=c, label=n)
                          for c, n in zip(CLASS_COLORS, CLASS_NAMES)]
        ax_db.legend(handles=legend_patches, fontsize=7, loc='lower right')

    # Confusion matrix
    ax_cm = axes[1, col]
    cm = confusion_matrix(y_test, y_pred_method)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax_cm,
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
                linewidths=0.5, linecolor='grey')
    ax_cm.set_xlabel('Predicted', fontsize=9)
    ax_cm.set_ylabel('True', fontsize=9)
    ax_cm.set_title(f'{method_name}\nConfusion Matrix', fontsize=9, fontweight='bold')
    plt.setp(ax_cm.get_xticklabels(), rotation=30, ha='right', fontsize=8)
    plt.setp(ax_cm.get_yticklabels(), rotation=0, fontsize=8)

plt.tight_layout()
plt.show()

print('Circles = training points | Stars = test points')
print('All three methods should produce very similar boundaries with the same kernel and C.')

## Section 13 — OvR vs OvO: When Does Each Win?

```
USE OvR WHEN:
  - K is large (K > 15) and test-time speed matters
  - Classes are well-separated (imbalance hurts less)
  - You need calibrated probabilities (OvR + Platt scaling works well)
  - Using LogisticRegression or other linear models (OvR is the natural default)

USE OvO WHEN:
  - Classes are similar in size but hard to separate pairwise
  - K is moderate (K < 20)
  - You're using SVM specifically (sklearn SVC uses OvO internally)
  - Class imbalance is severe (OvO's balanced pairs avoid this)

USE sklearn SVC (native) WHEN:
  - You want the best-optimised implementation
  - sklearn's libsvm handles memory and caching better than manual OvO
  - You need decision_function_shape for downstream probability calibration

DAGSVM:
  - OvO with test-time efficiency: O(K-1) evaluations vs O(K²)
  - Useful when K is large and inference latency matters
  - Not natively in sklearn — requires custom implementation
```

**Bottom line for email classification (4 classes):**  
Use `sklearn.svm.SVC` with `kernel='rbf'`, `decision_function_shape='ovr'`.  
It internally uses OvO (balanced training) but returns OvR-shaped decision functions for probability calibration.

## Self-Check Questions

Test your understanding before moving on.

---

**Question 1:**  
OvO trains C(K,2) binary SVMs while OvR trains K. For K=10 classes, OvO requires 45 classifiers compared to OvR's 10. Despite having 4.5× more classifiers, why might OvO still be preferred for SVM multi-class problems?

<details>
<summary>Show Answer</summary>

**OvO is often preferred for SVMs for several interconnected reasons:**

**1. Balanced training subsets (no class imbalance):**  
Each OvO classifier is trained on exactly 2 classes — a 50/50 split. OvR classifiers face a 1:(K-1) imbalance. For K=10, each OvR classifier trains on 10% positive vs 90% negative samples. This severe imbalance distorts the decision boundary toward the majority class and requires careful handling (class_weight='balanced'). OvO sidesteps this entirely.

**2. Smaller training sets per classifier — faster per-problem solve:**  
SVM training complexity is approximately O(n²) to O(n³) in the number of training samples. Each OvO classifier trains on ~2n/K samples. For K=10, that's ~n/5 samples each. Training cost per classifier is (n/5)² = n²/25 — about 25× faster per classifier. With 45 classifiers: 45 × n²/25 ≈ 1.8n², comparable to OvR's 10 × n² = 10n². In practice, OvO is often faster than OvR for SVMs because of this effect.

**3. Better binary separation:**  
Each OvO binary SVM only needs to separate two specific classes — a potentially easier problem than one class vs all others. This leads to tighter margins, fewer support vectors, and more accurate binary classifiers.

**4. sklearn's SVC uses OvO internally:**  
The default implementation in libsvm (used by sklearn's SVC) is OvO. This isn't accidental — the libsvm authors found OvO empirically competitive or superior to OvR on most benchmarks.

**5. Majority voting is robust:**  
The 45-vote ensemble is more robust to a single bad binary classifier than OvR's winner-takes-all with just 10 decisions.

**When OvR is still better:** When K > 50, OvO's quadratic growth in classifier count becomes impractical. For very large K, OvR or error-correcting output codes (ECOC) are preferred.

</details>

---

**Question 2:**  
In OvO, majority voting among C(K,2) pairwise classifiers determines the final class. Can you have a tie in OvO voting? How is it typically resolved? Give a concrete example for K=3 classes.

<details>
<summary>Show Answer</summary>

**Yes, ties can occur in OvO — and they're more common than you might expect.**

**Example for K=3, C(3,2) = 3 classifiers:**

Pairs: (A vs B), (A vs C), (B vs C)

If results are: A beats B, B beats C, C beats A  
→ Vote tally: A=1, B=1, C=1 — **three-way tie!** (This is a circular dominance pattern, like rock-paper-scissors)

**Tie-breaking strategies:**

1. **Decision function sum (sklearn's method):** Each binary classifier outputs a signed distance, not just ±1. Sum the decision function values for each class across all classifiers where it participated. The class with the highest total signed distance wins. This is more principled than pure vote counting.

2. **Random tie-breaking:** Pick a winner randomly among tied classes. Simple but non-deterministic.

3. **Lowest index wins:** Pick the class with the smallest index (label). Deterministic but arbitrary.

4. **DAG-SVM:** The directed acyclic graph structure eliminates ties by construction — each node in the tree either keeps or eliminates a class, so there's always a unique winner.

**sklearn's implementation:** Uses a combination of vote counting and decision function values as a tiebreaker. Specifically, `SVC` (using libsvm) computes the full vote matrix internally and uses `decision_function_shape` to reshape the output. Ties are resolved by the continuous decision function values, making the resolution principled.

**How often do ties occur?** For K=4 (our email problem), ties can occur when two classes each win 3 pairwise matches out of 6. Ties are more frequent when classes are very similar (small margin in the confused direction) — exactly when you need your classifier to be most careful.

</details>

---

**Question 3:**  
Your 4-class OvR email SVM achieves: Spam F1=0.95, Ham F1=0.91, Promotional F1=0.88, Newsletter F1=0.42. What is the most likely reason for the Newsletter class's very low F1 score? List at least two things you could try to fix it.

<details>
<summary>Show Answer</summary>

**Most likely reasons for Newsletter F1=0.42:**

**Primary cause: Class imbalance or class confusion**

1. **Class overlap:** Newsletters often contain promotional content, unsubscribe links (spam-like), and conversational text (ham-like). In feature space, newsletter emails likely overlap significantly with spam and promotional emails. The OvR classifier for "Newsletter vs Rest" faces a genuinely hard problem — it's trying to separate newsletters from 3 similar-looking classes.

2. **Severe imbalance in OvR:** The Newsletter vs Rest classifier trains with 25% positive (newsletters) vs 75% negative (everything else). With this 1:3 imbalance, the SVM may bias toward predicting "not newsletter" for most samples, giving high precision but low recall — which kills F1.

3. **Insufficient training data:** Newsletter may be underrepresented. If you have fewer newsletter examples, the SVM cannot learn a robust boundary.

**What to try:**

1. **`class_weight='balanced'`:** `SVC(class_weight='balanced')` — automatically adjusts the penalty C for each class inversely proportional to its frequency. This directly counteracts the 1:3 imbalance in OvR training. Often the single most effective fix.

2. **Switch from OvR to OvO:** OvO trains a balanced "Newsletter vs Spam", "Newsletter vs Ham", "Newsletter vs Promotional" — each is 50/50. This eliminates the imbalance problem entirely.

3. **Feature engineering:** Add newsletter-specific features — unsubscribe link presence, sender domain type, email list headers (List-Unsubscribe), HTML-heavy content ratio. Better features that discriminate newsletters will help any classifier.

4. **Collect more newsletter training data:** If newsletters are genuinely underrepresented, data collection is the most principled fix. Augment with email newsletter datasets (Enron, Apache SpamAssassin).

5. **Tune C independently:** Newsletter's poor F1 may stem from the global C being too aggressive. Use GridSearchCV with F1 scoring (not accuracy) to find C that optimises macro-F1 or newsletter-specific F1.

6. **Adjust decision threshold:** Lower the threshold for predicting "Newsletter" (below 0 in the decision function). This trades some precision for much better recall, which can recover F1 if the current issue is low recall.

7. **Examine confusion:** Print the row/column for Newsletter in the confusion matrix. If newsletter is mostly confused with "Promotional", that tells you exactly what feature engineering to add.

</details>